In [3]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score, KFold
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

from scipy.stats import norm
from scipy.spatial.distance import cdist
from scipy.stats.qmc import LatinHypercube

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Helper Functions

In [4]:
def compute_ucb(mu, sigma, kappa=2.5):
    return mu + kappa * sigma

def compute_ei(mu, sigma, f_best, xi=0.01):
    z = (mu - f_best - xi) / (sigma + 1e-9)
    ei = (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)
    ei[sigma == 0] = 0
    return ei

def format_query(point, decimals=6):
    point_clipped = np.clip(point, 0, 0.999999)
    return '-'.join([f'{x:.{decimals}f}' for x in point_clipped])

def thompson_sample(gp, candidates, n_samples=10, subsample=500):
    idx = np.random.choice(len(candidates), size=min(subsample, len(candidates)), replace=False)
    sub = candidates[idx]
    samples = gp.sample_y(sub, n_samples=n_samples, random_state=42)
    best_sub_idx = np.argmax(samples.mean(axis=1))
    return idx[best_sub_idx]

# Function 1 - Week 4

In [5]:
# =============================================================================
# FUNCTION 1 - Radiation Detection (2D)
# w3 output: -0.00161 -- first non-trivial signal
# changes: 
#    - drop the +-0.2 local radius, sweep the full space with LHS
#    - distance filter stops acquisition clustering near sampled dead zones
# =============================================================================

print("=" * 60)
print("Function 1 - Week 4")
print("=" * 60)

inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_1/initial_inputs.npy')
outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_1/initial_outputs.npy')

week1_query  = np.array([[0.959184, 0.836735]])
week1_output = np.array([-5.909566597235814e-107])
week2_query  = np.array([[0.374540, 0.950714]])
week2_output = np.array([-1.560646704467778e-117])
week3_query  = np.array([[0.694651, 0.629916]])
week3_output = np.array([-0.0016067678433140744])

all_inputs  = np.vstack([inputs, week1_query, week2_query, week3_query])
all_outputs = np.hstack([outputs, week1_output, week2_output, week3_output])

print(f"Total points: {len(all_outputs)}")
print(f"Current best: {all_outputs.max():.6e}")

log_outputs = np.log10(np.abs(all_outputs) + 1e-300)

kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
gp.fit(all_inputs, log_outputs)

# LHS gives better spatial coverage than uniform random
np.random.seed(42)
sampler    = LatinHypercube(d=2, seed=42)
candidates = sampler.random(n=10000)

mu, sigma = gp.predict(candidates, return_std=True)
ucb       = compute_ucb(mu, sigma, kappa=5.0)  # high kappa = strongly exploration-biased

# mask out anything too close to an existing sample
min_dist          = cdist(candidates, all_inputs).min(axis=1)
ucb[min_dist < 0.15] = -np.inf

best_idx = np.argmax(ucb)
query    = candidates[best_idx]

print(f"\nWeek 4 Query: {format_query(query)}")
print(f"mu: {mu[best_idx]:.4f}, sigma: {sigma[best_idx]:.4f}")
print(f"Min distance from existing samples: {min_dist[best_idx]:.4f}")

Function 1 - Week 4
Total points: 13
Current best: 7.710875e-16

Week 4 Query: 0.461537-0.459084
mu: -42.8030, sigma: 34.1597
Min distance from existing samples: 0.2890


# Function 2 - Week 4

In [6]:
# =============================================================================
# FUNCTION 2 - Noisy Log-Likelihood (2D)
# changes: 
#      - widen whitekernel bounds so the noise estimate isn't clamped
#      - use thompson sampling -- more robust when the GP mean is uncertain
# =============================================================================

print("\n" + "=" * 60)
print("Function 2 - Week 4")
print("=" * 60)

initial_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_2/initial_inputs.npy')
initial_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_2/initial_outputs.npy')

week1_input  = np.array([[0.775510, 0.959184]])
week1_output = np.array([0.16576674])
week2_input  = np.array([[0.683114, 0.932567]])
week2_output = np.array([0.56974583])
week3_input  = np.array([[0.794441, 0.256481]])
week3_output = np.array([0.27313450611629536])

all_inputs  = np.vstack([initial_inputs, week1_input, week2_input, week3_input])
all_outputs = np.hstack([initial_outputs, week1_output, week2_output, week3_output])

print(f"Total points: {len(all_outputs)}")
print(f"Current best: {all_outputs.max():.3f}")

# widen noise_level_bounds -- w3 hit lower bound (0.001), let it go lower
kernel = Matern(nu=2.5) + WhiteKernel(noise_level=0.05, noise_level_bounds=(1e-5, 1.0))
gp     = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=20, normalize_y=True)
gp.fit(all_inputs, all_outputs)
print(f"Learned kernel: {gp.kernel_}")

# LHS for better coverage, thompson sampling for robustness
sampler    = LatinHypercube(d=2, seed=42)
candidates = sampler.random(n=15000)

best_idx = thompson_sample(gp, candidates, n_samples=10)
query    = candidates[best_idx]

mu_q, sigma_q = gp.predict(query.reshape(1, -1), return_std=True)
print(f"\nWeek 4 Query: {format_query(query)}")
print(f"Predicted mean: {mu_q[0]:.3f}, std: {sigma_q[0]:.3f}")


Function 2 - Week 4
Total points: 13
Current best: 0.611
Learned kernel: Matern(length_scale=0.0549, nu=2.5) + WhiteKernel(noise_level=1e-05)

Week 4 Query: 0.706387-0.952221
Predicted mean: 0.523, std: 0.111


# Function 3 - Week 4

In [7]:
# =============================================================================
# FUNCTION 3 - Drug Discovery (3D)
# w3 output: -0.00063 (near zero -- best so far)
# w3 poly coefficients: C=+0.946, B=+0.484, A=-0.218
#   B^2=-1.291 (diminishing returns on B), B*C=+0.540, A*C=+0.414
# change: refit poly with w3 data, reduce B slightly, nudge C up
# blend GP EI + poly model 50/50, small xi (near optimum so exploit)
# =============================================================================

print("\n" + "=" * 60)
print("Function 3 - Week 4")
print("=" * 60)

f3_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_3/initial_inputs.npy')
f3_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_3/initial_outputs.npy')

week1_input  = np.array([[0.378956, 0.302768, 0.459346]])
week1_output = np.array([-0.02906213067759293])
week2_input  = np.array([[0.315339, 0.088659, 0.415174]])
week2_output = np.array([-0.06230989251412482])
week3_input  = np.array([[0.392735, 0.504381, 0.464332]])
week3_output = np.array([-0.0006328364393800658])

X = np.vstack([f3_inputs, week1_input, week2_input, week3_input])
Y = np.hstack([f3_outputs, week1_output, week2_output, week3_output])

print(f"Total points: {len(Y)}, best: {Y.max():.6f}")

poly       = PolynomialFeatures(degree=2, include_bias=False)
X_poly     = poly.fit_transform(X)
poly_model = LinearRegression()
poly_model.fit(X_poly, Y)

feature_names = ['A', 'B', 'C', 'A*B', 'A*C', 'B*C', 'A^2', 'B^2', 'C^2']
print("\nPolynomial coefficients (updated with w3):")
for name, coef in zip(feature_names, poly_model.coef_):
    print(f"  {name:6s}: {coef:+.4f}")

kernel = Matern(nu=2.5, length_scale=0.5) + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-4, 0.1))
gp     = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=20, normalize_y=True)
gp.fit(X, Y)
print(f"Learned kernel: {gp.kernel_}")

np.random.seed(42)

# local: tight around w3 best
local = []
for _ in range(5000):
    c = week3_input[0] + np.random.normal(0, 0.03, 3)
    local.append(np.clip(c, 0, 1))

# directed: reduce B (B^2 penalty), nudge C up (strong positive + interactions)
directed = []
for _ in range(5000):
    a = np.clip(week3_input[0, 0] + np.random.normal(0, 0.03), 0, 1)
    b = np.clip(week3_input[0, 1] - abs(np.random.normal(0, 0.08)), 0, 1)
    c = np.clip(week3_input[0, 2] + abs(np.random.normal(0, 0.06)), 0, 1)
    directed.append([a, b, c])

candidates = np.vstack([local, directed])

mu, sigma  = gp.predict(candidates, return_std=True)
ei         = compute_ei(mu, sigma, Y.max(), xi=0.001)  # small xi -- near optimum
poly_preds = poly_model.predict(poly.transform(candidates))

ei_norm   = (ei - ei.min()) / (ei.max() - ei.min() + 1e-9)
poly_norm = (poly_preds - poly_preds.min()) / (poly_preds.max() - poly_preds.min() + 1e-9)
blended   = 0.5 * ei_norm + 0.5 * poly_norm

best_idx = np.argmax(blended)
query    = candidates[best_idx]

print(f"\nWeek 4 Query: {format_query(query)}")
print(f"GP mean: {mu[best_idx]:.6f}, std: {sigma[best_idx]:.6f}")
print(f"Poly prediction: {poly_preds[best_idx]:.6f}")


Function 3 - Week 4
Total points: 18, best: -0.000633

Polynomial coefficients (updated with w3):
  A     : -0.1308
  B     : +0.6112
  C     : +1.1085
  A*B   : -0.1637
  A*C   : +0.3990
  B*C   : +0.5034
  A^2   : -0.3611
  B^2   : -1.2722
  C^2   : -0.8356
Learned kernel: Matern(length_scale=0.248, nu=2.5) + WhiteKernel(noise_level=0.0001)

Week 4 Query: 0.498607-0.467046-0.477827
GP mean: -0.018877, std: 0.036577
Poly prediction: -0.024100


# Function 4 - Week 4

In [8]:
# =============================================================================
# FUNCTION 4 - Warehouse Placement (4D)
# changes: 
#    - thompson sampling with global matern across full space
#    - goal: understand if the w2 area is a good area
# =============================================================================

print("\n" + "=" * 60)
print("Function 4 - Week 4")
print("=" * 60)

f4_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_4/initial_inputs.npy')
f4_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_4/initial_outputs.npy')

week1_input  = np.array([[0.466173, 0.451984, 0.359193, 0.383111]])
week1_output = np.array([-0.9654345395220925])
week2_input  = np.array([[0.424201, 0.406375, 0.372722, 0.413313]])
week2_output = np.array([0.6308582112564989])
week3_input  = np.array([[0.413541, 0.373697, 0.536536, 0.453354]])
week3_output = np.array([-2.1500998298742817])

all_inputs  = np.vstack([f4_inputs, week1_input, week2_input, week3_input])
all_outputs = np.hstack([f4_outputs, week1_output, week2_output, week3_output])

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.4f}")

# GP and matern -- had highest r2 in w3 cross-val
kernel = Matern(length_scale=0.5, length_scale_bounds=(0.001, 10.0), nu=2.5)
gp     = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15, normalize_y=True)
gp.fit(all_inputs, all_outputs)
print(f"Fitted kernel: {gp.kernel_}")

sampler    = LatinHypercube(d=4, seed=42)
candidates = sampler.random(n=20000)

# thompson sampling
best_idx = thompson_sample(gp, candidates, n_samples=10)
query    = candidates[best_idx]

mu_q, sigma_q = gp.predict(query.reshape(1, -1), return_std=True)
print(f"\nWeek 4 Query: {format_query(query)}")
print(f"Predicted mean: {mu_q[0]:.4f}, std: {sigma_q[0]:.4f}")
print(f"Distance from W2: {np.linalg.norm(query - week2_input[0]):.4f}")


Function 4 - Week 4
Total points: 33, best: 0.6309
Fitted kernel: Matern(length_scale=0.591, nu=2.5)

Week 4 Query: 0.424125-0.404716-0.487507-0.367688
Predicted mean: -1.0785, std: 0.6229
Distance from W2: 0.1235


# Function 5 - Week 4 

In [9]:
# =============================================================================
# FUNCTION 5 - Chemical Yield (4D)
# change: 
#    - push x1 lower (negative coef -- was drifting up) and x3 stays pinned at 0.999
#    - fix svr scaling: was predicting on raw scale, now uses standardscaler
# =============================================================================

print("\n" + "=" * 60)
print("Function 5 - Week 4")
print("=" * 60)

f5_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_5/initial_inputs.npy')
f5_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_5/initial_outputs.npy')

week1_query  = np.array([[0.284290, 0.869208, 0.999999, 0.903273]])
week1_output = np.array([2201.834589108927])
week2_query  = np.array([[0.311208, 0.881577, 0.999999, 0.915050]])
week2_output = np.array([2381.536867607932])
week3_query  = np.array([[0.355085, 0.899160, 0.999900, 0.934196]])
week3_output = np.array([2689.1537294933396])

all_inputs  = np.vstack([f5_inputs, week1_query, week2_query, week3_query])
all_outputs = np.hstack([f5_outputs, week1_output, week2_output, week3_output])

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.2f}")

lr = LinearRegression()
lr.fit(all_inputs, all_outputs)
print("\nLinear regression coefficients:")
for i, coef in enumerate(lr.coef_):
    print(f"  x{i+1}: {coef:+.2f}")

# svr with proper output scaling this time
input_scaler  = StandardScaler()
output_scaler = StandardScaler()
X_scaled = input_scaler.fit_transform(all_inputs)
Y_scaled = output_scaler.fit_transform(all_outputs.reshape(-1, 1)).ravel()

svr = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
svr.fit(X_scaled, Y_scaled)

kernel_gp = ConstantKernel(1.0, (0.001, 1000)) * Matern(length_scale=0.2, nu=2.5)
gp        = GaussianProcessRegressor(kernel=kernel_gp, n_restarts_optimizer=25, normalize_y=True)
gp.fit(all_inputs, all_outputs)

best_point = all_inputs[np.argmax(all_outputs)]
np.random.seed(42)

# local: small perturbations around w3 best
local = []
for _ in range(6000):
    c    = best_point + np.random.normal(0, 0.03, 4)
    c    = np.clip(c, 0, 1)
    c[2] = 0.999999  # x3 pinned
    local.append(c)

# directed: x1 down (negative coef), x2 and x4 up
directed = []
for _ in range(4000):
    c    = best_point.copy()
    c[0] = np.clip(c[0] - abs(np.random.normal(0, 0.05)), 0, 1)
    c[1] = np.clip(c[1] + abs(np.random.normal(0, 0.03)), 0, 1)
    c[2] = 0.999999
    c[3] = np.clip(c[3] + abs(np.random.normal(0, 0.03)), 0, 1)
    directed.append(c)

candidates = np.vstack([local, directed])

# svr pre-filter
X_cand_scaled = input_scaler.transform(candidates)
svr_preds_s   = svr.predict(X_cand_scaled)
svr_preds     = output_scaler.inverse_transform(svr_preds_s.reshape(-1, 1)).ravel()

top_idx  = np.argsort(svr_preds)[-3000:]
filtered = candidates[top_idx]

mu, sigma = gp.predict(filtered, return_std=True)
ei        = compute_ei(mu, sigma, all_outputs.max(), xi=0.005)

best_idx = np.argmax(ei)
query    = filtered[best_idx]

print(f"\nWeek 4 Query: {format_query(query)}")
print(f"GP mean: {mu[best_idx]:.2f}, std: {sigma[best_idx]:.2f}")
print(f"SVR prediction: {svr_preds[top_idx[best_idx]]:.2f}")
print(f"EI: {ei[best_idx]:.6f}")



Function 5 - Week 4
Total points: 23, best: 2689.15

Linear regression coefficients:
  x1: -145.40
  x2: +811.09
  x3: +1529.96
  x4: +942.30

Week 4 Query: 0.453609-0.922659-0.999999-0.960267
GP mean: 3213.33, std: 81.98
SVR prediction: 2952.21
EI: 524.175192


# Function 6 - Week 4

In [10]:
# =============================================================================
# FUNCTION 6 - Cake Recipe (5D)
# changes: 
#     - vary x1 freely (most sensitive per ARD), hold x3 and x4 tight
#     - x2 and x5 get moderate variation
# =============================================================================

print("\n" + "=" * 60)
print("Function 6 - Week 4")
print("=" * 60)

f6_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_6/initial_inputs.npy')
f6_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_6/initial_outputs.npy')

prev_queries = np.array([
    [0.201818, 0.220639, 0.476414, 0.867793, 0.023115],
    [0.457382, 0.349799, 0.529520, 0.691032, 0.192589],
    [0.503404, 0.328861, 0.662838, 0.995031, 0.025514],
])
prev_outputs = np.array([-0.792246, -0.361637, -0.3676262724593105])

all_inputs  = np.vstack([f6_inputs, prev_queries])
all_outputs = np.hstack([f6_outputs, prev_outputs])

print(f"Total observations: {all_inputs.shape[0]}, best: {all_outputs.max():.4f}")

lr = LinearRegression()
lr.fit(all_inputs, all_outputs)
print("\nLinear regression coefficients:")
for i, coef in enumerate(lr.coef_):
    print(f"  x{i+1}: {coef:+.4f}")

kernel = ConstantKernel(1.0, (0.001, 1000)) * Matern(
    length_scale=[1.0, 1.0, 1.0, 1.0, 1.0],
    length_scale_bounds=(0.05, 5.0),
    nu=2.5
)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=25, normalize_y=True)
gp.fit(all_inputs, all_outputs)

fitted_ls = gp.kernel_.k2.length_scale
print("\nFitted ARD length scales (smaller = more sensitive):")
for i, ls in enumerate(fitted_ls):
    print(f"  x{i+1}: {ls:.3f}")

best_point = all_inputs[np.argmax(all_outputs)]
np.random.seed(42)

candidates = []
for _ in range(15000):
    c    = best_point.copy()
    c[0] = np.random.uniform(0, 1)           # x1: free (most sensitive)
    c[1] += np.random.normal(0, 0.10)        # x2: moderate
    c[2] += np.random.normal(0, 0.05)        # x3: small (low sensitivity)
    c[3] += np.random.normal(0, 0.05)        # x4: small (low sensitivity)
    c[4] += np.random.normal(0, 0.10)        # x5: moderate
    candidates.append(np.clip(c, 0, 1))

candidates = np.array(candidates)

mu, sigma = gp.predict(candidates, return_std=True)
ei        = compute_ei(mu, sigma, all_outputs.max(), xi=0.02)

best_idx = np.argmax(ei)
query    = candidates[best_idx]

print(f"\nWeek 4 Query: {format_query(query)}")
print(f"Predicted mean: {mu[best_idx]:.4f}, std: {sigma[best_idx]:.4f}")
print(f"EI: {ei[best_idx]:.6f}")



Function 6 - Week 4
Total observations: 23, best: -0.3616

Linear regression coefficients:
  x1: -0.4709
  x2: -0.5619
  x3: +0.3492
  x4: +0.6728
  x5: -0.8991

Fitted ARD length scales (smaller = more sensitive):
  x1: 0.635
  x2: 1.100
  x3: 1.467
  x4: 1.301
  x5: 0.943

Week 4 Query: 0.484266-0.300747-0.678836-0.776480-0.244496
Predicted mean: -0.3307, std: 0.0866
EI: 0.040269


# Function 7 - Week 4

In [11]:
# =============================================================================
# FUNCTION 7 - ML Hyperparameters (6D)
# changes: 
#     - add MLP as a second surrogate, push hp5 lower and hp6 higher
#     - query where GP EI and MLP both agree
# =============================================================================

print("\n" + "=" * 60)
print("Function 7 - Week 4")
print("=" * 60)

f7_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_7/initial_inputs.npy')
f7_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_7/initial_outputs.npy')

week1_query  = np.array([[0.045091, 0.528666, 0.329265, 0.105350, 0.434671, 0.641164]])
week1_output = np.array([1.0510006614196026])
week2_query  = np.array([[0.034389, 0.427542, 0.236649, 0.342956, 0.405706, 0.695527]])
week2_output = np.array([1.6531363312716738])
week3_query  = np.array([[0.020000, 0.062635, 0.587368, 0.313225, 0.362008, 0.664539]])
week3_output = np.array([2.6016443512251484])

all_inputs  = np.vstack([f7_inputs, week1_query, week2_query, week3_query])
all_outputs = np.hstack([f7_outputs, week1_output, week2_output, week3_output])

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.4f}")

# re-check directions with w3 data included
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(all_inputs, all_outputs)

feature_names = [f"HP{i+1}" for i in range(6)]
print("\nRF feature importance (updated):")
for name, imp in zip(feature_names, rf.feature_importances_):
    print(f"  {name}: {imp:.3f}")

top_mask     = all_outputs >= np.percentile(all_outputs, 75)
dataset_mean = all_inputs.mean(axis=0)
top_mean     = all_inputs[top_mask].mean(axis=0)
print("\nDirectional signals:")
for name, tm, dm in zip(feature_names, top_mean, dataset_mean):
    print(f"  {name}: top {tm:.3f} vs mean {dm:.3f} -> {'LOW' if tm < dm else 'HIGH'}")

# GP on log-transformed outputs (same as w3)
kernel = ConstantKernel(1.0, (0.01, 10.0)) * Matern(
    length_scale=[0.3] * 6,
    length_scale_bounds=(0.05, 5.0),
    nu=2.5
)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=20, normalize_y=True)
gp.fit(all_inputs, np.log1p(all_outputs))

# MLP surrogate 
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_sc     = scaler_x.fit_transform(all_inputs)
Y_sc     = scaler_y.fit_transform(all_outputs.reshape(-1, 1)).ravel()

mlp = MLPRegressor(
    hidden_layer_sizes=(32, 32),
    activation='relu',
    alpha=0.01,       # L2 regularisation
    max_iter=2000,
    random_state=42
)
mlp.fit(X_sc, Y_sc)

# candidates: hp1 stays near 0, push hp5 lower, push hp6 higher than w3
np.random.seed(42)
n          = 20000
candidates = np.zeros((n, 6))
candidates[:, 0] = np.random.uniform(0.0, 0.10, n)   # hp1 low
candidates[:, 1] = np.random.uniform(0.0, 1.0,  n)   # hp2 low importance
candidates[:, 2] = np.random.uniform(0.0, 1.0,  n)   # hp3 low importance
candidates[:, 3] = np.random.uniform(0.0, 1.0,  n)   # hp4 low importance
candidates[:, 4] = np.random.uniform(0.0, 0.25, n)   # hp5 lower than w3's 0.362
candidates[:, 5] = np.random.uniform(0.7, 1.0,  n)   # hp6 higher than w3's 0.665

# GP EI
best_log    = np.log1p(all_outputs.max())
mu_gp, s_gp = gp.predict(candidates, return_std=True)
ei_gp       = compute_ei(mu_gp, s_gp, best_log, xi=0.01)

# MLP predictions
X_cand_sc   = scaler_x.transform(candidates)
mlp_sc      = mlp.predict(X_cand_sc)
mlp_preds   = scaler_y.inverse_transform(mlp_sc.reshape(-1, 1)).ravel()

# combine -- normalise both and average
ei_norm  = (ei_gp   - ei_gp.min())   / (ei_gp.max()   - ei_gp.min()   + 1e-9)
mlp_norm = (mlp_preds - mlp_preds.min()) / (mlp_preds.max() - mlp_preds.min() + 1e-9)
combined = 0.5 * ei_norm + 0.5 * mlp_norm

best_idx = np.argmax(combined)
query    = candidates[best_idx]

print(f"\nWeek 4 Query: {format_query(query)}")
print(f"GP mean (log): {mu_gp[best_idx]:.4f}, std: {s_gp[best_idx]:.4f}")
print(f"MLP prediction: {mlp_preds[best_idx]:.4f}")
print(f"EI: {ei_gp[best_idx]:.6f}")


Function 7 - Week 4
Total points: 33, best: 2.6016

RF feature importance (updated):
  HP1: 0.769
  HP2: 0.055
  HP3: 0.031
  HP4: 0.015
  HP5: 0.076
  HP6: 0.054

Directional signals:
  HP1: top 0.230 vs mean 0.467 -> LOW
  HP2: top 0.392 vs mean 0.391 -> HIGH
  HP3: top 0.511 vs mean 0.389 -> HIGH
  HP4: top 0.462 vs mean 0.489 -> LOW
  HP5: top 0.319 vs mean 0.461 -> LOW
  HP6: top 0.655 vs mean 0.501 -> HIGH

Week 4 Query: 0.012096-0.033240-0.640160-0.336533-0.245422-0.901043
GP mean (log): 1.1181, std: 0.1577
MLP prediction: 2.7043
EI: 0.010840


# Function 8 - Week 4

In [12]:
# =============================================================================
# FUNCTION 8 - 8D Black-Box
# changes: 
#      - add MLP as second surrogate alongside GP
#      - bias dimentsions: x1 and x6 back toward w1 values, x3 free (most sensitive), x7 free
#      - use thompson + mlp agreement to pick query
# =============================================================================

print("\n" + "=" * 60)
print("Function 8 - Week 4")
print("=" * 60)

f8_inputs  = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_8/initial_inputs.npy')
f8_outputs = np.load('/Users/odes_to_knavery/Documents/AI_ML/Captone/bbo_data/function_8/initial_outputs.npy')

week1_query  = np.array([[0.182943, 0.000000, 0.000000, 0.000000, 0.999999, 0.127124, 0.092403, 0.000000]])
week1_output = np.array([9.6723503773075])
week2_query  = np.array([[0.022695, 0.000000, 0.274348, 0.000000, 0.999999, 0.127124, 0.028034, 0.000000]])
week2_output = np.array([9.6263579169495])
week3_query  = np.array([[0.005086, 0.000000, 0.048567, 0.000000, 0.999999, 0.998962, 0.018674, 0.000000]])
week3_output = np.array([9.5463675507445])

all_inputs  = np.vstack([f8_inputs, week1_query, week2_query, week3_query])
all_outputs = np.hstack([f8_outputs, week1_output, week2_output, week3_output])

print(f"Total points: {len(all_outputs)}, best: {all_outputs.max():.4f} (W1)")

# ARD GP
kernel = ConstantKernel(1.0, (0.001, 100.0)) * Matern(
    length_scale=np.ones(8),
    length_scale_bounds=(0.01, 100.0),
    nu=2.5
)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=20, normalize_y=True)
gp.fit(all_inputs, all_outputs)

ls = gp.kernel_.k2.length_scale
print("\nLearned ARD length scales (smaller = more sensitive):")
for i, l in enumerate(ls):
    print(f"  x{i+1}: {l:.2f}")

# MLP surrogate
scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_sc     = scaler_x.fit_transform(all_inputs)
Y_sc     = scaler_y.fit_transform(all_outputs.reshape(-1, 1)).ravel()

mlp = MLPRegressor(
    hidden_layer_sizes=(32, 32),
    activation='relu',
    alpha=0.01,
    max_iter=2000,
    random_state=42
)
mlp.fit(X_sc, Y_sc)

np.random.seed(42)
n    = 25000
pool = np.zeros((n, 8))

# pins: same as w3
pool[:, 1] = 0.0
pool[:, 3] = 0.0
pool[:, 4] = 0.999999
pool[:, 7] = 0.0

# free dimensions -- pull x1 and x6 back toward w1 values
# w1 had x1=0.183, x6=0.127 -- both drifted in w2 and w3
pool[:, 0] = np.clip(np.random.normal(0.183, 0.10, n), 0, 1)  # x1 near w1
pool[:, 2] = np.random.uniform(0, 1, n)                         # x3 free (most sensitive)
pool[:, 5] = np.clip(np.random.normal(0.127, 0.15, n), 0, 1)  # x6 near w1
pool[:, 6] = np.random.uniform(0, 1, n)                         # x7 free

# GP UCB scores
gp_mu, gp_sigma = gp.predict(pool, return_std=True)
gp_ucb          = compute_ucb(gp_mu, gp_sigma, kappa=2.0)

# MLP scores
X_pool_sc  = scaler_x.transform(pool)
mlp_sc     = mlp.predict(X_pool_sc)
mlp_preds  = scaler_y.inverse_transform(mlp_sc.reshape(-1, 1)).ravel()

# combine and query where both agree
gp_norm  = (gp_ucb   - gp_ucb.min())   / (gp_ucb.max()   - gp_ucb.min()   + 1e-9)
mlp_norm = (mlp_preds - mlp_preds.min()) / (mlp_preds.max() - mlp_preds.min() + 1e-9)
combined = 0.5 * gp_norm + 0.5 * mlp_norm

best_idx = np.argmax(combined)
query    = pool[best_idx]

print(f"\nWeek 4 Query: {format_query(query)}")
print(f"GP mu: {gp_mu[best_idx]:.4f}, sigma: {gp_sigma[best_idx]:.4f}")
print(f"MLP prediction: {mlp_preds[best_idx]:.4f}")



Function 8 - Week 4
Total points: 43, best: 9.6724 (W1)

Learned ARD length scales (smaller = more sensitive):
  x1: 4.38
  x2: 6.55
  x3: 3.54
  x4: 7.41
  x5: 21.24
  x6: 6.45
  x7: 4.42
  x8: 100.00

Week 4 Query: 0.000000-0.000000-0.012582-0.000000-0.999999-0.000000-0.066411-0.000000
GP mu: 9.5342, sigma: 0.0842
MLP prediction: 9.9687
